# Phase 1 — PP-OCRv5_server_det: kiểm chứng checkpoint chính thức

**Đề tài:** Robust Scene Text Detection under Realistic Image Degradation
**Phạm vi notebook:** chỉ detector `PP-OCRv5_server_det` (PaddleOCR 3.x), không recognition, không train/fine-tune, không đổi threshold.

| Mục | Nội dung |
|---|---|
| Số paper/official | Hmean **83,8%** trên bộ đánh giá riêng của PaddleOCR 3.0 (2.677 ảnh) |
| Có reproduce đúng 83,8 không? | Chỉ khi có đúng ảnh + GT + evaluator gốc. Không có → ghi **NOT VERIFIED** |
| Notebook này chạy | **External validation**: checkpoint chính thức, cấu hình mặc định, trên **ICDAR2015 test (500 ảnh)**, chấm bằng giao thức IoU kiểu ICDAR/RRC (logic giống `DetectionIoUEvaluator` của PaddleOCR) |
| Kết luận được phép | "official-checkpoint validation / external benchmark" — **không** gọi là exact reproduction |

### Trước khi bấm Run All trên Kaggle
1. **Settings → Accelerator: GPU T4** (1 GPU là đủ; Phase 1 chạy single-GPU theo handoff).
2. **Settings → Internet: On** (cần cho `pip install`, tải checkpoint, `git ls-remote`). Tài khoản phải **xác minh số điện thoại** thì mới bật được; khi bấm *Save Version* thì Internet cũng phải đang On. Cell kiểm tra mạng ở mục 1 sẽ dừng ngay nếu thiếu mạng.
3. **Add Input → Datasets** (không phải *Notebooks*): gắn https://www.kaggle.com/datasets/bestofbests9/icdar2015 (Version 3, 136.35 MB; có `Detection/ch4_test_images` = 500 ảnh `img_*.jpg` và `Detection/ch4_test_localization_transcription_gt` = 500 file `gt_img_*.txt`). Gắn một *notebook* thì Kaggle chỉ mount output của notebook đó, không có ảnh. Nên dùng **cùng dataset với Tuấn** để hai model chấm trên cùng bytes.
   - Notebook tự dò thư mục; nếu dò sai/nhập nhằng thì điền tay `ICDAR15_TEST_IMG_DIR` / `ICDAR15_TEST_GT_DIR` ở cell Config.
   - Cũng chấp nhận file nhãn kiểu PaddleOCR `test_icdar2015_label.txt` (`ICDAR15_PADDLE_LABEL`).
4. Chạy lần lượt; **đọc output cell Smoke test (mục 6) trước khi chạy full**.

Đầu ra: `/kaggle/working/artifacts/phase1_paddle/` + file zip `phase1_paddle_artifacts.zip` (đúng cây thư mục `artifacts/phase1_paddle/` trong handoff).

## 0. Config

In [ ]:
# ==== CONFIG — chỉ chỉnh ở cell này ====
import os, sys, re, json, time, hashlib, platform, subprocess, shutil, random, datetime
from pathlib import Path
from collections import Counter

MODEL_NAME = 'PP-OCRv5_server_det'
DEVICE = 'gpu:0'                       # Phase 1: single GPU. Đổi 'cpu' chỉ khi bản GPU không cài được (ghi lại lý do)
# Kaggle image hiện là Python 3.13: paddlepaddle-gpu chỉ có wheel cp313 từ bản 3.2.0 (3.0.x không cài được)
PADDLE_PKG = 'paddlepaddle-gpu==3.3.1'
PADDLE_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'
PADDLE_CPU_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cpu/'
PADDLEOCR_PKG = 'paddleocr==3.3.3'
PADDLE_WHEEL_DIR = None                # dự phòng: thư mục dataset chứa file paddlepaddle_gpu-*.whl tự upload

# Dữ liệu ICDAR2015 test — để None cho notebook tự dò trong /kaggle/input
ICDAR15_TEST_IMG_DIR = None            # vd '/kaggle/input/<dataset>/ch4_test_images'
ICDAR15_TEST_GT_DIR = None             # vd '/kaggle/input/<dataset>/Challenge4_Test_Task1_GT'
ICDAR15_PADDLE_LABEL = None            # thay cho GT dir: file test_icdar2015_label.txt (format PaddleOCR)
ALLOW_RRC_DOWNLOAD = False             # True = tải trực tiếp từ rrc.cvc.uab.es (phải chấp nhận điều khoản ICDAR)
OFFICIAL_BENCH_DIR = None              # bộ 2.677 ảnh của PaddleOCR 3.0 nếu nhóm có được; None = không có

# Giao thức chấm (cố định trước khi xem kết quả)
EXPECTED_N_IMAGES = 500
IOU_THR = 0.5                          # match khi IoU > 0.5 (strict, như RRC/PaddleOCR)
IGNORE_AREA_PRECISION_THR = 0.5        # det bị bỏ qua khi > 50% diện tích nằm trong GT '###'
IGNORE_TAGS = ('###', '*')
SWEEP_SCORE_THRS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]   # chỉ diagnostic, KHÔNG dùng làm số chính

N_SMOKE = 20
N_VIS = 20
N_DETERMINISM = 10
N_WARMUP = 3
SEED = 2026

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
ART = WORK / 'artifacts' / 'phase1_paddle'
VIS_DIR = ART / 'visual_examples'
RRC_DIR = WORK / 'icdar2015_rrc'
for d in (ART, VIS_DIR):
    d.mkdir(parents=True, exist_ok=True)
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat()


def sh(cmd, log=None, check=True):
    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True)
    out = p.stdout + p.stderr
    if log is not None:
        with open(log, 'a', encoding='utf-8') as f:
            f.write(f'$ {cmd}\n{out}\n')
    if check and p.returncode != 0:
        print(out[-3000:])
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')
    return out


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf-8')


print('Artifacts ->', ART)

## 1. Preflight: GPU / Python / disk

In [ ]:
print(sh('nvidia-smi', check=False))
print('python', sys.version)
print(sh('df -h /kaggle/working', check=False))

In [ ]:
# Kiểm tra mạng trước khi cài — báo lỗi rõ thay vì để pip retry vài phút
import socket, urllib.request, urllib.error

NET_URLS = {
    'pypi.org': 'https://pypi.org/simple/pip/',
    'www.paddlepaddle.org.cn': PADDLE_INDEX,
    'github.com': 'https://github.com',
    'huggingface.co': 'https://huggingface.co',
    'paddle-model-ecology.bj.bcebos.com': 'https://paddle-model-ecology.bj.bcebos.com',
}
net = {}
for host, url in NET_URLS.items():
    r = {'dns': False, 'https': False, 'error': None}
    try:
        socket.getaddrinfo(host, 443)
        r['dns'] = True
        try:
            urllib.request.urlopen(url, timeout=20).close()
            r['https'] = True
        except urllib.error.HTTPError:
            r['https'] = True          # server trả lời (403/404...) = vẫn tới được
    except Exception as e:
        r['error'] = f'{type(e).__name__}: {e}'
    net[host] = r
    print(f'{host:38s} dns={r["dns"]!s:5s} https={r["https"]!s:5s} {r["error"] or ""}')
write_json(ART / 'network_check.json', net)

if not net['pypi.org']['https']:
    raise RuntimeError('Notebook KHÔNG có Internet. Bật: thanh bên phải → Settings → Internet: On '
                       '(tài khoản Kaggle phải xác minh số điện thoại), rồi Run All / Save Version lại.')
if not net['www.paddlepaddle.org.cn']['https'] and not PADDLE_WHEEL_DIR:
    raise RuntimeError('Có Internet nhưng không tới được www.paddlepaddle.org.cn (PyPI không có paddlepaddle-gpu 3.x). '
                       'Dùng phương án wheel tự upload: xem hướng dẫn ở mục 2, rồi đặt PADDLE_WHEEL_DIR.')
if not (net['huggingface.co']['https'] or net['paddle-model-ecology.bj.bcebos.com']['https']):
    print('WARN: không tới được nguồn tải checkpoint (HuggingFace/BOS) — bước nạp model có thể lỗi')

In [ ]:
# Kiểm tra nhanh dữ liệu TRƯỚC khi cài (bước cài mất ~12 phút) — chỉ dùng thư viện chuẩn
ICDAR15_DATASET_URL = 'https://www.kaggle.com/datasets/bestofbests9/icdar2015'


def quick_data_check(root=INPUT_ROOT, n=EXPECTED_N_IMAGES):
    want = set(range(1, n + 1))
    img_hits, gt_hits, label_hits, nb_outputs = [], [], [], []
    for dp, dn, fn in os.walk(root):
        ids_img = {int(m.group(1)) for f in fn if (m := re.fullmatch(r'img_(\d+)\.(?:jpg|jpeg|png)', f, re.I))}
        ids_gt = {int(m.group(1)) for f in fn if (m := re.fullmatch(r'gt_img_(\d+)\.txt', f))}
        if ids_img == want:
            img_hits.append(dp)
        if ids_gt == want:
            gt_hits.append(dp)
        label_hits += [os.path.join(dp, f) for f in fn if re.fullmatch(r'.*test.*label.*\.txt', f, re.I)]
        if '__notebook__.ipynb' in fn:
            nb_outputs.append(dp)
    return img_hits, gt_hits, label_hits, nb_outputs


if ICDAR15_TEST_IMG_DIR or ALLOW_RRC_DOWNLOAD:
    print('Bỏ qua quick check (đã đặt đường dẫn tay hoặc bật ALLOW_RRC_DOWNLOAD)')
else:
    img_hits, gt_hits, label_hits, nb_outputs = quick_data_check()
    print('ảnh test (img_1..500):', img_hits)
    print('GT test (gt_img_1..500):', gt_hits, '| label PaddleOCR:', label_hits)
    if not img_hits or not (gt_hits or label_hits or ICDAR15_TEST_GT_DIR or ICDAR15_PADDLE_LABEL):
        hint = ''
        if nb_outputs:
            hint = (f'\nĐang gắn một NOTEBOOK ({nb_outputs[0]}): Kaggle chỉ mount output của notebook đó, KHÔNG kèm dữ liệu. '
                    f'Gỡ nó ra, rồi Add Input → Datasets → {ICDAR15_DATASET_URL}')
        raise RuntimeError(f'Chưa có ICDAR2015 test trong {INPUT_ROOT}.{hint}\n'
                           f'Gợi ý dataset: {ICDAR15_DATASET_URL} (Version 3, 136.35 MB, thư mục Detection/ch4_test_images).')
    print('quick data check OK')

## 2. Cài PaddlePaddle + PaddleOCR

Cài đúng theo cách docs chính thức (index riêng của PaddlePaddle cho CUDA 12.6; PyPI không có `paddlepaddle-gpu` 3.x). Log cài đặt lưu ở `pip_install.log`.

Nếu cài GPU lỗi: **không** thử bừa nhiều phiên bản — lưu log, ghi lại lỗi, rồi mới quyết định.

**Dự phòng khi Kaggle không tới được `www.paddlepaddle.org.cn`:** trên máy mình tải wheel cho Linux / Python 3.13:
```
pip download paddlepaddle-gpu==3.3.1 --no-deps --only-binary=:all: --python-version 3.13 --platform linux_x86_64 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ -d paddle_wheel
```
rồi upload thư mục `paddle_wheel` thành một Kaggle Dataset (private), gắn vào notebook và đặt `PADDLE_WHEEL_DIR='/kaggle/input/<dataset>'`. Notebook ghi sha256 của wheel vào `env_versions.txt`.

In [ ]:
INSTALL_LOG = ART / 'pip_install.log'
FREEZE_BEFORE = sh([sys.executable, '-m', 'pip', 'freeze'], check=False)
(ART / 'pip_freeze_before_install.txt').write_text(FREEZE_BEFORE, encoding='utf-8')
if PADDLE_WHEEL_DIR:
    wheels = sorted(Path(PADDLE_WHEEL_DIR).rglob('paddlepaddle*.whl'))
    if len(wheels) != 1:
        raise RuntimeError(f'Cần đúng 1 wheel paddlepaddle trong {PADDLE_WHEEL_DIR}, thấy {[w.name for w in wheels]}')
    sh([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])], log=INSTALL_LOG)
    PADDLE_SOURCE = f'local wheel {wheels[0].name} sha256={sha256_file(wheels[0])}'
elif DEVICE.startswith('gpu'):
    sh([sys.executable, '-m', 'pip', 'install', '-q', PADDLE_PKG, '-i', PADDLE_INDEX], log=INSTALL_LOG)
    PADDLE_SOURCE = f'pip install {PADDLE_PKG} -i {PADDLE_INDEX}'
else:
    cpu_pkg = PADDLE_PKG.replace('-gpu', '')
    sh([sys.executable, '-m', 'pip', 'install', '-q', cpu_pkg, '-i', PADDLE_CPU_INDEX], log=INSTALL_LOG)
    PADDLE_SOURCE = f'pip install {cpu_pkg} -i {PADDLE_CPU_INDEX}'
sh([sys.executable, '-m', 'pip', 'install', '-q', PADDLEOCR_PKG, 'shapely'], log=INSTALL_LOG)
print('install OK — log:', INSTALL_LOG)

In [ ]:
# paddlepaddle-gpu kéo theo các gói nvidia-* (cu12.6) và có thể hạ cấp NCCL mà torch có sẵn của Kaggle cần.
# `import paddleocr` -> paddlex -> modelscope -> import torch -> lỗi `undefined symbol: ncclCommShrink`.
# Pipeline Paddle không dùng torch: torch hỏng thì gỡ khỏi phiên Kaggle này và ghi lại (không đụng tới Paddle).
import importlib, importlib.util


def _pkg_lines(freeze):
    return {ln.split('==')[0].lower(): ln for ln in freeze.splitlines()
            if '==' in ln and ln.lower().startswith(('nvidia-', 'torch'))}


freeze_after = sh([sys.executable, '-m', 'pip', 'freeze'], check=False)
before, after = _pkg_lines(FREEZE_BEFORE), _pkg_lines(freeze_after)
changes = [f'{k}: {before.get(k, "-")} -> {after.get(k, "-")}' for k in sorted(set(before) | set(after))
           if before.get(k) != after.get(k)]
(ART / 'nvidia_torch_pkg_changes.txt').write_text('\n'.join(changes) + '\n', encoding='utf-8')
print('nvidia-*/torch thay đổi khi cài Paddle:', *changes, sep='\n  ')

TORCH_ACTION = 'torch không có sẵn'
if importlib.util.find_spec('torch') is not None:
    p = subprocess.run([sys.executable, '-c', 'import torch'], capture_output=True, text=True)
    if p.returncode == 0:
        TORCH_ACTION = 'torch vẫn import được — giữ nguyên'
    else:
        err = (p.stderr.strip().splitlines() or ['?'])[-1]
        sh([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], log=INSTALL_LOG)
        importlib.invalidate_caches()
        assert importlib.util.find_spec('torch') is None, 'gỡ torch chưa xong'
        TORCH_ACTION = (f'torch hỏng sau khi cài Paddle ({err}) -> đã gỡ torch/torchvision/torchaudio '
                        f'khỏi phiên Kaggle này; pipeline Paddle không dùng torch')
print(TORCH_ACTION)

In [ ]:
# paddlex import sẵn các gói TUỲ CHỌN nếu chúng được cài (chỉ kiểm tra "có cài", không kiểm tra version),
# vd langchain>=1.0 của Kaggle không còn `langchain.docstore` trong khi paddlex cần langchain<1.0.
# Thử `import paddleocr` ở tiến trình riêng; nếu lỗi do một gói paddlex coi là tuỳ chọn (không thuộc ocr-core)
# thì gỡ gói đó khỏi phiên này rồi thử lại. Lỗi do gói khác -> dừng, không gỡ bừa.
import importlib.metadata as md
from packaging.requirements import Requirement

PROBE = r"""
import json, traceback
try:
    import paddleocr
    r = {'ok': True}
except Exception as e:
    files = [f.filename for f in traceback.extract_tb(e.__traceback__)]
    r = {'ok': False, 'type': type(e).__name__, 'msg': str(e)[:300], 'name': getattr(e, 'name', None), 'files': files}
print('PROBE_RESULT:' + json.dumps(r))
"""


def canon(name):
    return re.sub(r'[-_.]+', '-', name).lower()


def removable_optional_deps():
    reqs = [Requirement(r) for r in (md.requires('paddlex') or [])]
    is_extra = lambda r: r.marker is not None and 'extra' in str(r.marker)
    base = {canon(r.name) for r in reqs if not is_extra(r)}
    ocr_core = {canon(r.name) for r in reqs if is_extra(r) and r.marker.evaluate({'extra': 'ocr-core'})}
    return {canon(r.name) for r in reqs if is_extra(r)} - base - ocr_core - {'paddlepaddle', 'paddleocr', 'paddlex'}


def top_from_file(path):
    m = re.search(r'[/\\](?:site|dist)-packages[/\\]([^/\\]+)', path or '')
    return m.group(1).removesuffix('.py') if m else None


def culprit_dists(res, removable):
    pkg_dists = md.packages_distributions()
    cands = []
    if res.get('name'):
        cands.append(res['name'].split('.')[0])
    cands += [t for t in (top_from_file(f) for f in reversed(res.get('files', [])))
              if t and t not in ('paddlex', 'paddleocr', 'paddle')]
    for top in cands:
        bad = sorted({canon(d) for d in pkg_dists.get(top, [])} & removable)
        if bad:
            return bad
    return []


REMOVABLE = removable_optional_deps()
REMOVED_OPTIONAL = []
for attempt in range(6):
    p = subprocess.run([sys.executable, '-c', PROBE], capture_output=True, text=True)
    line = next((ln for ln in reversed(p.stdout.splitlines()) if ln.startswith('PROBE_RESULT:')), None)
    if line is None:
        raise RuntimeError(f'probe import paddleocr không trả kết quả (exit {p.returncode}):\n{p.stderr[-2000:]}')
    res = json.loads(line.removeprefix('PROBE_RESULT:'))
    if res['ok']:
        break
    bad = culprit_dists(res, REMOVABLE)
    if not bad:
        raise RuntimeError(f'import paddleocr lỗi do gói không thuộc nhóm tuỳ chọn của paddlex — cần xem tay: '
                           f'{res["type"]}: {res["msg"]}\nfiles: {res["files"][-4:]}')
    sh([sys.executable, '-m', 'pip', 'uninstall', '-y', *bad], log=INSTALL_LOG)
    importlib.invalidate_caches()
    REMOVED_OPTIONAL.append({'uninstalled': bad, 'because': f'{res["type"]}: {res["msg"]}'})
    print(f'gỡ {bad} vì: {res["type"]}: {res["msg"]}')
else:
    raise RuntimeError(f'import paddleocr vẫn lỗi sau 6 lần gỡ: {REMOVED_OPTIONAL}')
write_json(ART / 'removed_optional_deps.json', REMOVED_OPTIONAL)
print('import paddleocr OK (probe) | đã gỡ:', REMOVED_OPTIONAL or 'không có')

In [ ]:
import paddle
print('paddle', paddle.__version__, '| commit', paddle.version.commit)
print('compiled with CUDA:', paddle.is_compiled_with_cuda(), '| cuda', paddle.version.cuda(), '| cudnn', paddle.version.cudnn())
if DEVICE.startswith('gpu'):
    assert paddle.is_compiled_with_cuda(), 'Paddle không có CUDA — kiểm tra lại bước cài'
    print('visible GPUs:', paddle.device.cuda.device_count())
paddle.utils.run_check()

In [ ]:
# env_versions.txt + git_commit.txt
import importlib.metadata as md


def pkg_ver(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


env = {
    'utc_time': RUN_STARTED,
    'python': sys.version,
    'platform': platform.platform(),
    'device': DEVICE,
    'paddlepaddle': paddle.__version__,
    'paddle_commit': paddle.version.commit,
    'paddle_cuda': paddle.version.cuda(),
    'paddle_cudnn': paddle.version.cudnn(),
    'paddleocr': pkg_ver('paddleocr'),
    'paddlex': pkg_ver('paddlex'),
    'numpy': pkg_ver('numpy'),
    'opencv': pkg_ver('opencv-contrib-python') or pkg_ver('opencv-python') or pkg_ver('opencv-python-headless'),
    'shapely': pkg_ver('shapely'),
    'gpu': sh('nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader', check=False).strip(),
    'install_cmds': [PADDLE_SOURCE, f'pip install {PADDLEOCR_PKG} shapely'],
    'torch_action': TORCH_ACTION,
    'removed_optional_deps': REMOVED_OPTIONAL,
    'PADDLE_PDX_MODEL_SOURCE': os.environ.get('PADDLE_PDX_MODEL_SOURCE'),
}
freeze = sh([sys.executable, '-m', 'pip', 'freeze'], check=False)
(ART / 'env_versions.txt').write_text(
    '\n'.join(f'{k}: {v}' for k, v in env.items()) + '\n\n# pip freeze\n' + freeze, encoding='utf-8')


def tag_commit(repo, version):
    # commit của tag phát hành tương ứng với bản pip (ưu tiên commit đã peel của annotated tag)
    if not version:
        return 'NOT INSTALLED'
    tag = f'refs/tags/v{version}'
    out = sh(['git', 'ls-remote', '--tags', f'https://github.com/{repo}.git'], check=False)
    refs = dict((ln.split('\t')[1], ln.split('\t')[0]) for ln in out.splitlines() if '\t' in ln)
    return refs.get(tag + '^{}') or refs.get(tag) or f'NOT FOUND ({tag})'


git_lines = [
    f'PaddleOCR  pip={env["paddleocr"]}  tag_commit={tag_commit("PaddlePaddle/PaddleOCR", env["paddleocr"])}',
    f'PaddleX    pip={env["paddlex"]}  tag_commit={tag_commit("PaddlePaddle/PaddleX", env["paddlex"])}',
    f'Paddle     pip={env["paddlepaddle"]}  build_commit={env["paddle_commit"]}',
    'Note: cài từ pip wheel, không clone repo; commit = commit của git tag ứng với phiên bản pip.',
]
(ART / 'git_commit.txt').write_text('\n'.join(git_lines) + '\n', encoding='utf-8')
print(json.dumps(env, indent=2, default=str))
print('\n'.join(git_lines))

## 3. Inspect `/kaggle/input` (không đoán cấu trúc thư mục)

Xem output này để chắc dataset ICDAR2015 đã được gắn và đúng split test.

In [ ]:
def summarize_tree(root, max_depth=4):
    root = Path(root)
    if not root.exists():
        print('missing', root)
        return
    for dp, dn, fn in os.walk(root):
        dn.sort()
        depth = len(Path(dp).relative_to(root).parts)
        if depth > max_depth:
            dn[:] = []
            continue
        exts = Counter(Path(f).suffix.lower() for f in fn)
        print('  ' * depth + f'{Path(dp).name}/  files={len(fn)} {dict(exts.most_common(4))}  e.g. {sorted(fn)[:3]}')


summarize_tree(INPUT_ROOT)

In [ ]:
# (Tuỳ chọn) tải ICDAR2015 test trực tiếp từ RRC — chỉ khi không có dataset Kaggle và nhóm đồng ý điều khoản ICDAR
RRC_URLS = {
    'ch4_test_images.zip': 'https://rrc.cvc.uab.es/downloads/ch4_test_images.zip',
    'Challenge4_Test_Task1_GT.zip': 'https://rrc.cvc.uab.es/downloads/Challenge4_Test_Task1_GT.zip',
}
if ALLOW_RRC_DOWNLOAD:
    import urllib.request, zipfile
    RRC_DIR.mkdir(parents=True, exist_ok=True)
    for name, url in RRC_URLS.items():
        dst = RRC_DIR / name
        if not dst.exists():
            urllib.request.urlretrieve(url, dst)
        print(name, 'sha256', sha256_file(dst))
        with zipfile.ZipFile(dst) as z:
            z.extractall(RRC_DIR / Path(name).stem)
else:
    print('ALLOW_RRC_DOWNLOAD=False — dùng dataset đã gắn trong /kaggle/input')

## 4. Core: dò dữ liệu, parse GT, evaluator

Evaluator là giao thức IoU kiểu ICDAR2015/RRC, cùng logic `ppocr/metrics/eval_det_iou.py::DetectionIoUEvaluator`:
- GT có transcription `###` là *don't care*; polygon không hợp lệ bị bỏ (và được đếm, báo cáo).
- Prediction bị bỏ qua nếu > 50% diện tích của nó nằm trong một GT don't care.
- Ghép 1-1 theo thứ tự (greedy), điều kiện IoU **> 0,5**; P/R/Hmean là micro trên toàn bộ 500 ảnh.

In [ ]:
# == CORE ==
import numpy as np
from shapely.geometry import Polygon

ID_RE_GT = re.compile(r'gt_img_(\d+)\.txt')
ID_RE_IMG = re.compile(r'img_(\d+)\.(?:jpg|jpeg|png)', re.I)
LABEL_RE = re.compile(r'.*test.*label.*\.txt', re.I)


def scan_dirs(roots):
    gt_dirs, img_dirs, label_files = {}, {}, []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for dp, dn, fn in os.walk(root):
            g = {int(m.group(1)) for f in fn if (m := ID_RE_GT.fullmatch(f))}
            i = {int(m.group(1)) for f in fn if (m := ID_RE_IMG.fullmatch(f))}
            if g:
                gt_dirs[Path(dp)] = g
            if i:
                img_dirs[Path(dp)] = i
            label_files += [Path(dp) / f for f in fn if LABEL_RE.fullmatch(f)]
    return gt_dirs, img_dirs, label_files


def pick_one(cands, what):
    if len(cands) == 1:
        return cands[0]
    test_like = [c for c in cands if 'test' in str(c).lower()]
    if len(test_like) == 1:
        return test_like[0]
    raise RuntimeError(f'{what}: {len(cands)} ứng viên, hãy điền tay trong CONFIG: {[str(c) for c in cands]}')


def resolve_icdar15(roots, n_expected=EXPECTED_N_IMAGES):
    want = set(range(1, n_expected + 1))
    gt_dirs, img_dirs, label_files = scan_dirs(roots)
    if ICDAR15_TEST_IMG_DIR:
        img_dir = Path(ICDAR15_TEST_IMG_DIR)
    else:
        cands = [d for d, ids in img_dirs.items() if ids == want]
        if not cands:
            raise RuntimeError(f'Không tìm thấy thư mục chứa đúng img_1..img_{n_expected}. Các thư mục ảnh thấy được: '
                               f'{[(str(d), len(ids)) for d, ids in img_dirs.items()][:10]}')
        img_dir = pick_one(cands, 'ICDAR2015 test image dir')
    if ICDAR15_PADDLE_LABEL:
        return img_dir, Path(ICDAR15_PADDLE_LABEL), 'paddle_label'
    if ICDAR15_TEST_GT_DIR:
        return img_dir, Path(ICDAR15_TEST_GT_DIR), 'rrc'
    gt_cands = [d for d, ids in gt_dirs.items() if ids == want]
    if gt_cands:
        return img_dir, pick_one(gt_cands, 'ICDAR2015 test GT dir'), 'rrc'
    if label_files:
        return img_dir, pick_one(label_files, 'PaddleOCR label file'), 'paddle_label'
    raise RuntimeError('Không tìm thấy GT test (gt_img_*.txt hoặc *test*label*.txt). Điền tay trong CONFIG.')


def parse_rrc_gt_file(path):
    # dòng RRC: x1,y1,...,x4,y4,transcription (transcription có thể chứa dấu phẩy)
    items = []
    with open(path, encoding='utf-8-sig') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip().lstrip(chr(0xFEFF))
            if not line:
                continue
            parts = line.split(',')
            if len(parts) < 9:
                raise ValueError(f'{path}:{ln}: cần 8 toạ độ + text, nhận {line!r}')
            pts = np.array([float(x) for x in parts[:8]], dtype=np.float64).reshape(4, 2)
            text = ','.join(parts[8:])
            items.append({'points': pts, 'text': text, 'ignore': text in IGNORE_TAGS})
    return items


def parse_paddle_label(path):
    gts = {}
    with open(path, encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            rel, js = line.rstrip('\n').split('\t', 1)
            m = ID_RE_IMG.fullmatch(Path(rel).name)
            if not m:
                raise ValueError(f'{path}:{ln}: tên ảnh lạ {rel!r}')
            gts[int(m.group(1))] = [
                {'points': np.asarray(a['points'], dtype=np.float64).reshape(-1, 2),
                 'text': a['transcription'], 'ignore': a['transcription'] in IGNORE_TAGS}
                for a in json.loads(js)]
    return gts


def load_icdar15(img_dir, gt_src, fmt, n_expected=EXPECTED_N_IMAGES):
    img_dir, gt_src = Path(img_dir), Path(gt_src)
    imgs = {int(m.group(1)): img_dir / f for f in os.listdir(img_dir) if (m := ID_RE_IMG.fullmatch(f))}
    if fmt == 'rrc':
        gts = {int(m.group(1)): parse_rrc_gt_file(gt_src / f)
               for f in os.listdir(gt_src) if (m := ID_RE_GT.fullmatch(f))}
    else:
        gts = parse_paddle_label(gt_src)
    if set(imgs) != set(gts):
        raise ValueError(f'Lệch id ảnh/GT: chỉ có ảnh={sorted(set(imgs) - set(gts))[:10]}, '
                         f'chỉ có GT={sorted(set(gts) - set(imgs))[:10]}')
    if len(imgs) != n_expected:
        raise ValueError(f'Có {len(imgs)} ảnh, kỳ vọng {n_expected} (ICDAR2015 test)')
    return [{'image_id': f'img_{k}', 'path': str(imgs[k]), 'gt': gts[k]} for k in sorted(imgs)]


def _valid_poly(pts):
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 3 or not np.all(np.isfinite(pts)):
        return None
    p = Polygon(pts)
    return p if p.is_valid else None


def evaluate_image(gt_items, det_polys, iou_thr=IOU_THR, ign_thr=IGNORE_AREA_PRECISION_THR):
    gt_pols, gt_ign, det_pols, det_ign = [], [], [], []
    n_invalid_gt = n_invalid_det = 0
    for g in gt_items:
        p = _valid_poly(g['points'])
        if p is None:
            n_invalid_gt += 1
            continue
        gt_pols.append(p)
        if g['ignore']:
            gt_ign.append(len(gt_pols) - 1)
    for d in det_polys:
        p = _valid_poly(d)
        if p is None:
            n_invalid_det += 1
            continue
        det_pols.append(p)
        for gi in gt_ign:
            if p.area > 0 and gt_pols[gi].intersection(p).area / p.area > ign_thr:
                det_ign.append(len(det_pols) - 1)
                break
    gt_ign_s, det_ign_s = set(gt_ign), set(det_ign)
    gt_hit = np.zeros(len(gt_pols), dtype=bool)
    det_hit = np.zeros(len(det_pols), dtype=bool)
    pairs = []
    if gt_pols and det_pols:
        iou = np.zeros((len(gt_pols), len(det_pols)))
        for i, g in enumerate(gt_pols):
            for j, d in enumerate(det_pols):
                inter = g.intersection(d).area
                if inter > 0:
                    iou[i, j] = inter / g.union(d).area
        for i in range(len(gt_pols)):
            for j in range(len(det_pols)):
                if (not gt_hit[i] and not det_hit[j] and i not in gt_ign_s and j not in det_ign_s
                        and iou[i, j] > iou_thr):
                    gt_hit[i] = det_hit[j] = True
                    pairs.append((i, j, float(iou[i, j])))
    return {
        'n_gt_care': len(gt_pols) - len(gt_ign), 'n_det_care': len(det_pols) - len(det_ign),
        'matched': len(pairs), 'n_gt_ignore': len(gt_ign), 'n_det_ignore': len(det_ign),
        'n_invalid_gt': n_invalid_gt, 'n_invalid_det': n_invalid_det, 'pairs': pairs,
        'gt_pols': gt_pols, 'det_pols': det_pols, 'gt_ign': gt_ign_s, 'det_ign': det_ign_s,
        'gt_hit': gt_hit, 'det_hit': det_hit,
    }


def combine(results):
    m = sum(r['matched'] for r in results)
    g = sum(r['n_gt_care'] for r in results)
    d = sum(r['n_det_care'] for r in results)
    P = m / d if d else 0.0
    R = m / g if g else 0.0
    H = 2 * P * R / (P + R) if P + R else 0.0
    return {'precision': P, 'recall': R, 'hmean': H, 'matched': m, 'gt_care': g, 'det_care': d,
            'gt_ignore': sum(r['n_gt_ignore'] for r in results),
            'det_ignore': sum(r['n_det_ignore'] for r in results),
            'invalid_gt': sum(r['n_invalid_gt'] for r in results),
            'invalid_det': sum(r['n_invalid_det'] for r in results)}

In [ ]:
# Dò + nạp ICDAR2015 test, ghi dataset_manifest.tsv (sha256 từng ảnh + GT) để đối chiếu với Tuấn
from PIL import Image

roots = [INPUT_ROOT] + ([RRC_DIR] if RRC_DIR.exists() else [])
IMG_DIR, GT_SRC, GT_FMT = resolve_icdar15(roots)
samples = load_icdar15(IMG_DIR, GT_SRC, GT_FMT)
print('images :', IMG_DIR)
print('GT     :', GT_SRC, f'(format={GT_FMT})')

sizes = Counter()
rows = ['image_id\tsha256\twidth\theight\tn_gt\tn_gt_ignore\tpath']
for s in samples:
    with Image.open(s['path']) as im:
        s['size'] = im.size
        exif_orient = im.getexif().get(0x0112)
    if exif_orient not in (None, 1):
        print('WARN EXIF orientation', exif_orient, s['image_id'])
    sizes[s['size']] += 1
    n_ign = sum(g['ignore'] for g in s['gt'])
    rows.append(f'{s["image_id"]}\t{sha256_file(s["path"])}\t{s["size"][0]}\t{s["size"][1]}\t{len(s["gt"])}\t{n_ign}\t{s["path"]}')
(ART / 'dataset_manifest.tsv').write_text('\n'.join(rows) + '\n', encoding='utf-8')
DATASET_SHA = sha256_file(ART / 'dataset_manifest.tsv')

n_gt = sum(len(s['gt']) for s in samples)
n_ign = sum(g['ignore'] for s in samples for g in s['gt'])
print(f'{len(samples)} ảnh | {n_gt} GT boxes ({n_ign} là ###, {n_gt - n_ign} care) | kích thước ảnh: {dict(sizes)}')
print('dataset_manifest.tsv sha256:', DATASET_SHA)

## 5. Nạp checkpoint chính thức + ghi hash/config

Không truyền tham số override nào (`limit_side_len`, `thresh`, `box_thresh`, `unclip_ratio`... giữ mặc định của model). Hash toàn bộ file trong thư mục model → `checkpoint_sha256.txt`.

In [ ]:
from paddleocr import TextDetection

t0 = time.perf_counter()
det = TextDetection(model_name=MODEL_NAME, device=DEVICE)
MODEL_LOAD_S = time.perf_counter() - t0
print(f'model loaded in {MODEL_LOAD_S:.1f}s')


def find_model_dir():
    cache = os.environ.get('PADDLE_PDX_CACHE_HOME')
    bases = ([Path(cache)] if cache else []) + [Path.home() / '.paddlex', Path('/root/.paddlex'), WORK]
    for base in bases:
        p = base / 'official_models' / MODEL_NAME
        if (p / 'inference.yml').exists():
            return p
    for base in bases:
        if base.exists():
            for dp, dn, fn in os.walk(base):
                if Path(dp).name == MODEL_NAME and 'inference.yml' in fn:
                    return Path(dp)
    raise FileNotFoundError(f'Không thấy thư mục model {MODEL_NAME} (inference.yml)')


MODEL_DIR = find_model_dir()
ck_lines = []
for f in sorted(p for p in MODEL_DIR.rglob('*') if p.is_file() and '.cache' not in p.relative_to(MODEL_DIR).parts):
    ck_lines.append(f'{sha256_file(f)}  {f.stat().st_size:>12}  {f.relative_to(MODEL_DIR)}')
(ART / 'checkpoint_sha256.txt').write_text(
    f'model: {MODEL_NAME}\nmodel_dir: {MODEL_DIR}\nsource: tải tự động bởi paddleocr {env["paddleocr"]} / paddlex {env["paddlex"]}\n\n'
    + '\n'.join(ck_lines) + '\n', encoding='utf-8')
shutil.copy(MODEL_DIR / 'inference.yml', ART / 'inference.yml')
print((ART / 'checkpoint_sha256.txt').read_text())

import yaml
with open(MODEL_DIR / 'inference.yml', encoding='utf-8') as f:
    INFER_CFG = yaml.safe_load(f)
INFER_PARAMS = {
    'overrides_passed_to_TextDetection': {},
    'PreProcess': INFER_CFG.get('PreProcess'),
    'PostProcess': INFER_CFG.get('PostProcess'),
    'batch_size': 1,
}
write_json(ART / 'inference_params.json', INFER_PARAMS)
print(json.dumps(INFER_PARAMS, indent=2, default=str))

## 6. Smoke test 20 ảnh — kiểm tra `dt_polys`, `dt_scores`

Dừng lại đọc output: số polygon hợp lý, 4 đỉnh/polygon, score trong [0,1], toạ độ nằm trong ảnh. Nếu adapter lỗi key → sửa theo API của phiên bản đang chạy, **không** đổi checkpoint/threshold.

In [ ]:
def extract(res):
    try:
        d = {'dt_polys': res['dt_polys'], 'dt_scores': res['dt_scores']}
    except (KeyError, TypeError):
        j = res.json
        j = j.get('res', j)
        d = {'dt_polys': j['dt_polys'], 'dt_scores': j['dt_scores']}
    polys = [np.asarray(p, dtype=np.float64).reshape(-1, 2) for p in d['dt_polys']]
    scores = [float(s) for s in d['dt_scores']]
    if len(polys) != len(scores):
        raise ValueError(f'polygon/score mismatch {len(polys)} vs {len(scores)}')
    return polys, scores


def predict_one(path):
    out = list(det.predict(input=path, batch_size=1))
    if len(out) != 1:
        raise RuntimeError(f'{path}: predict trả {len(out)} kết quả')
    return extract(out[0]), out[0]


rng = random.Random(SEED)
smoke = sorted(rng.sample(range(len(samples)), N_SMOKE))
for k, idx in enumerate(smoke):
    s = samples[idx]
    (polys, scores), raw = predict_one(s['path'])
    W, H = s['size']
    nv = Counter(len(p) for p in polys)
    oob = sum(int((p[:, 0].min() < -1) or (p[:, 1].min() < -1) or (p[:, 0].max() > W + 1) or (p[:, 1].max() > H + 1)) for p in polys)
    srange = f'[{min(scores):.3f}, {max(scores):.3f}]' if scores else '-'
    print(f'{s["image_id"]:>8} {W}x{H} | gt={len(s["gt"]):2d} | det={len(polys):2d} vertices={dict(nv)} score={srange} out_of_bounds={oob}')
    assert all(np.all(np.isfinite(p)) for p in polys)
    assert all(0.0 <= x <= 1.0 for x in scores)
    if k == 0:
        write_json(ART / 'smoke_raw_example.json', raw.json)
print('smoke test OK')

## 7. Full inference ICDAR2015 test (single GPU, batch_size=1)

In [ ]:
for s in samples[:N_WARMUP]:
    predict_one(s['path'])

preds, lat_ms = {}, {}
t_all = time.perf_counter()
for k, s in enumerate(samples):
    t = time.perf_counter()
    (polys, scores), _ = predict_one(s['path'])
    lat_ms[s['image_id']] = (time.perf_counter() - t) * 1000
    preds[s['image_id']] = {
        'sample_id': f'icdar15_test::{s["image_id"]}', 'image_id': s['image_id'], 'condition': 'icdar2015_test',
        'model': 'paddle', 'model_name': MODEL_NAME,
        'polygons': [[float(v) for v in p.reshape(-1)] for p in polys], 'scores': scores,
    }
    if (k + 1) % 100 == 0:
        print(f'{k + 1}/{len(samples)}  {time.perf_counter() - t_all:.0f}s')
assert set(preds) == {s['image_id'] for s in samples}

PRED_PATH = ART / 'paddle_icdar2015_test_predictions.jsonl'
with PRED_PATH.open('w', encoding='utf-8') as f:
    for s in samples:
        f.write(json.dumps(preds[s['image_id']], ensure_ascii=False) + '\n')
lat = np.array([lat_ms[s['image_id']] for s in samples])
LATENCY = {'mean_ms': float(lat.mean()), 'median_ms': float(np.median(lat)), 'p90_ms': float(np.percentile(lat, 90)),
           'p95_ms': float(np.percentile(lat, 95)), 'total_s': float(lat.sum() / 1000),
           'note': 'gồm đọc ảnh + preprocess + forward + postprocess, sau warmup, batch_size=1'}
print('predictions ->', PRED_PATH, '| sha256', sha256_file(PRED_PATH))
print(json.dumps(LATENCY, indent=2))

In [ ]:
# Chạy lại N ảnh đầu: kiểm tra tính tất định (ghi nhận, không sửa kết quả)
det_check = []
for s in samples[:N_DETERMINISM]:
    (polys, scores), _ = predict_one(s['path'])
    a = preds[s['image_id']]
    same_n = len(polys) == len(a['polygons'])
    max_poly = max((float(np.abs(p.reshape(-1) - np.asarray(q)).max()) for p, q in zip(polys, a['polygons'])), default=0.0) if same_n else None
    max_score = max((abs(x - y) for x, y in zip(scores, a['scores'])), default=0.0) if same_n else None
    det_check.append({'image_id': s['image_id'], 'same_count': same_n, 'max_abs_poly_diff': max_poly, 'max_abs_score_diff': max_score})
DETERMINISM = {'n_images': len(det_check), 'all_same_count': all(r['same_count'] for r in det_check),
               'max_abs_poly_diff': max((r['max_abs_poly_diff'] or 0) for r in det_check),
               'max_abs_score_diff': max((r['max_abs_score_diff'] or 0) for r in det_check), 'details': det_check}
print(json.dumps({k: v for k, v in DETERMINISM.items() if k != 'details'}, indent=2))

## 8. Đánh giá — P / R / Hmean

- **Số chính (primary):** dự đoán ở cấu hình mặc định, không lọc thêm.
- **Sweep score threshold:** chỉ diagnostic, vì chọn threshold trên chính tập test là lạc quan. Một số repo (vd MMOCR `HmeanIOUMetric`) báo Hmean tốt nhất theo sweep — ghi lại để Tuấn đối chiếu cách báo cáo, **không** dùng làm số chính. Mặc định của model đã lọc `box_thresh`, nên các ngưỡng thấp hơn nó cho kết quả giống nhau.

In [ ]:
import pandas as pd

eval_results = {}
for s in samples:
    p = preds[s['image_id']]
    eval_results[s['image_id']] = evaluate_image(s['gt'], [np.asarray(x).reshape(-1, 2) for x in p['polygons']])
PRIMARY = combine(list(eval_results.values()))

SWEEP = {}
for thr in SWEEP_SCORE_THRS:
    res = []
    for s in samples:
        p = preds[s['image_id']]
        keep = [np.asarray(x).reshape(-1, 2) for x, sc in zip(p['polygons'], p['scores']) if sc >= thr]
        res.append(evaluate_image(s['gt'], keep))
    SWEEP[str(thr)] = combine(res)
best_thr = max(SWEEP, key=lambda t: SWEEP[t]['hmean'])

per_rows = []
for s in samples:
    r = eval_results[s['image_id']]
    P = r['matched'] / r['n_det_care'] if r['n_det_care'] else None
    R = r['matched'] / r['n_gt_care'] if r['n_gt_care'] else None
    per_rows.append({'image_id': s['image_id'], 'gt_care': r['n_gt_care'], 'gt_ignore': r['n_gt_ignore'],
                     'det_care': r['n_det_care'], 'det_ignore': r['n_det_ignore'], 'matched': r['matched'],
                     'precision': P, 'recall': R, 'invalid_det': r['n_invalid_det'], 'latency_ms': lat_ms[s['image_id']]})
per_df = pd.DataFrame(per_rows)
per_df.to_csv(ART / 'per_image_eval.csv', index=False)

METRICS = {
    'model': MODEL_NAME,
    'evaluation_type': 'external_benchmark_validation',
    'benchmark': {'name': 'ICDAR2015 Incidental Scene Text, test split', 'n_images': len(samples),
                  'image_dir': str(IMG_DIR), 'gt_source': str(GT_SRC), 'gt_format': GT_FMT,
                  'dataset_manifest_sha256': DATASET_SHA},
    'protocol': {'name': 'ICDAR2015 RRC IoU (logic như PaddleOCR DetectionIoUEvaluator)', 'iou_thr_strict_gt': IOU_THR,
                 'ignore_area_precision_thr': IGNORE_AREA_PRECISION_THR, 'ignore_tags': list(IGNORE_TAGS),
                 'matching': 'greedy one-to-one', 'aggregation': 'micro over all images'},
    'primary': PRIMARY,
    'score_threshold_sweep_diagnostic': {'results': SWEEP, 'best_thr': best_thr, 'best_hmean': SWEEP[best_thr]['hmean'],
                                         'warning': 'threshold chọn trên tập test — không dùng làm số chính'},
    'inference': INFER_PARAMS,
    'latency': LATENCY,
    'determinism_check': {k: v for k, v in DETERMINISM.items() if k != 'details'},
    'predictions_file': PRED_PATH.name, 'predictions_sha256': sha256_file(PRED_PATH),
    'env': {k: env[k] for k in ('paddlepaddle', 'paddleocr', 'paddlex', 'paddle_cuda', 'gpu', 'device')},
    'official_reference': {'hmean': 0.838, 'benchmark': 'PaddleOCR 3.0 internal detection set (2,677 images)',
                           'comparable_with_this_run': False},
    'utc_time': RUN_STARTED,
}
write_json(ART / 'metrics.json', METRICS)

print(f'PRIMARY  P={PRIMARY["precision"]:.4f}  R={PRIMARY["recall"]:.4f}  Hmean={PRIMARY["hmean"]:.4f}  '
      f'(matched={PRIMARY["matched"]}, gt_care={PRIMARY["gt_care"]}, det_care={PRIMARY["det_care"]})')
print(f'ignored: gt={PRIMARY["gt_ignore"]} det={PRIMARY["det_ignore"]} | invalid polygons: gt={PRIMARY["invalid_gt"]} det={PRIMARY["invalid_det"]}')
print(pd.DataFrame(SWEEP).T[['precision', 'recall', 'hmean', 'det_care']].round(4))

## 9. Visual examples (20 ảnh)

Màu: **xanh lá** = GT được match · **cam** = GT bị bỏ sót (FN) · **xám** = GT `###` / det bị ignore · **xanh dương** = det đúng (TP) · **đỏ** = det sai (FP).
Chọn ảnh: một nửa là các ảnh có Hmean thấp nhất (để xem lỗi), một nửa ngẫu nhiên (seed cố định).

In [ ]:
import cv2

COL = {'gt_tp': (0, 200, 0), 'gt_fn': (0, 165, 255), 'ign': (160, 160, 160), 'det_tp': (255, 80, 0), 'det_fp': (0, 0, 255)}


def ring(p):
    return np.asarray(p.exterior.coords, dtype=np.float64)[:-1].round().astype(np.int32).reshape(-1, 1, 2)


def draw(sample, r):
    img = cv2.imread(sample['path'])
    for i, p in enumerate(r['gt_pols']):
        c = COL['ign'] if i in r['gt_ign'] else (COL['gt_tp'] if r['gt_hit'][i] else COL['gt_fn'])
        cv2.polylines(img, [ring(p)], True, c, 3)
    for j, p in enumerate(r['det_pols']):
        c = COL['ign'] if j in r['det_ign'] else (COL['det_tp'] if r['det_hit'][j] else COL['det_fp'])
        cv2.polylines(img, [ring(p)], True, c, 1)
    txt = f'{sample["image_id"]}  TP={r["matched"]} GT={r["n_gt_care"]} DET={r["n_det_care"]}'
    cv2.putText(img, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 4)
    cv2.putText(img, txt, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    return img


def f1(row):
    p, r = row['precision'], row['recall']
    if p is None or r is None or pd.isna(p) or pd.isna(r):
        return float('nan')
    return 2 * p * r / (p + r) if p + r else 0.0


per_df['hmean'] = per_df.apply(f1, axis=1)
worst = per_df[per_df['gt_care'] > 0].sort_values(['hmean', 'image_id']).head(N_VIS // 2)['image_id'].tolist()
rest = sorted(set(per_df['image_id']) - set(worst))
chosen = worst + sorted(random.Random(SEED).sample(rest, N_VIS - len(worst)))
by_id = {s['image_id']: s for s in samples}
for iid in chosen:
    tag = 'worst' if iid in worst else 'random'
    cv2.imwrite(str(VIS_DIR / f'{tag}_{iid}.jpg'), draw(by_id[iid], eval_results[iid]))
per_df[per_df['image_id'].isin(chosen)].assign(group=lambda d: d['image_id'].map(lambda x: 'worst' if x in worst else 'random')) \
    .to_csv(VIS_DIR / 'index.csv', index=False)
print(len(chosen), 'ảnh ->', VIS_DIR)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, iid in zip(axes.ravel(), worst[:3] + chosen[N_VIS // 2:N_VIS // 2 + 3]):
    ax.imshow(cv2.cvtColor(draw(by_id[iid], eval_results[iid]), cv2.COLOR_BGR2RGB))
    ax.set_title(iid)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Official benchmark availability + evaluation type

In [ ]:
if OFFICIAL_BENCH_DIR:
    raise NotImplementedError('Đã có OFFICIAL_BENCH_DIR: ghi rõ format ảnh/GT/evaluator gốc trước, rồi mới viết code chấm — không chấm bằng evaluator tự chế.')

P, R, H = PRIMARY['precision'], PRIMARY['recall'], PRIMARY['hmean']
availability_md = f"""# PaddleOCR 3.0 official detection benchmark — availability

- **Số cần reproduce:** `{MODEL_NAME}` Hmean **83,8%** trên bộ đánh giá text detection riêng của PaddleOCR 3.0 (2.677 ảnh, nhiều nguồn).
  Nguồn: https://www.paddleocr.ai/main/en/version3.x/module_usage/text_detection.html · tech report https://arxiv.org/abs/2507.05595
- **Trạng thái exact reproduction 83,8%:** **NOT VERIFIED**
- **Lý do:** lần chạy này không có ảnh + GT + evaluator gốc của bộ 2.677 ảnh (`OFFICIAL_BENCH_DIR=None`).

## Đã kiểm tra (điền tay trước khi nộp — ghi link + ngày)
- [ ] Trang docs Text Detection (link ở trên): có link tải bộ đánh giá không?
- [ ] Tech report arXiv 2507.05595: có công bố dữ liệu/evaluator không?
- [ ] GitHub PaddleOCR issues/discussions:
- [ ] Hỏi maintainer (nếu có):

## Thay thế đã chạy (cần nhóm đồng ý bằng văn bản trước khi sang Phase 2)
- Checkpoint chính thức, cấu hình mặc định, không fine-tune/không đổi threshold.
- Benchmark công khai: ICDAR2015 test, {len(samples)} ảnh. Evaluator: IoU kiểu RRC (logic `DetectionIoUEvaluator` của PaddleOCR), IoU > {IOU_THR}.
- Kết quả: **P {P*100:.2f} · R {R*100:.2f} · Hmean {H*100:.2f}** (%).
- Con số này **không so trực tiếp** với 83,8% (khác dữ liệu, khác phân bố, khác annotation).
"""
(ART / 'official_benchmark_availability.md').write_text(availability_md, encoding='utf-8')
(ART / 'evaluation_type.txt').write_text(
    'external\n'
    'official-checkpoint validation on a public external benchmark (ICDAR2015 test, 500 images).\n'
    'NOT an exact reproduction of the PaddleOCR 3.0 official Hmean 83.8 (official 2,677-image set not available here).\n',
    encoding='utf-8')
print(availability_md)

## 11. Đóng gói artifacts

In [ ]:
listing = []
for f in sorted(p for p in ART.rglob('*') if p.is_file() and p.name != 'artifacts_sha256.txt'):
    listing.append(f'{sha256_file(f)}  {f.relative_to(ART.parent)}')
(ART / 'artifacts_sha256.txt').write_text('\n'.join(listing) + '\n', encoding='utf-8')
zip_path = shutil.make_archive(str(WORK / 'phase1_paddle_artifacts'), 'zip', root_dir=ART.parent.parent, base_dir='artifacts')
print('zip ->', zip_path, '| sha256', sha256_file(zip_path))
print()
print('| Model | Benchmark | #images | Precision | Recall | Hmean | mean latency (ms) | Loại |')
print('|---|---|---:|---:|---:|---:|---:|---|')
print(f'| {MODEL_NAME} | ICDAR2015 test | {len(samples)} | {P*100:.2f} | {R*100:.2f} | {H*100:.2f} | {LATENCY["mean_ms"]:.1f} | external validation |')
print(f'| {MODEL_NAME} | PaddleOCR 3.0 internal (2.677) | — | — | — | 83.8 (official) | — | NOT VERIFIED |')

## Gate 1 — phần Paddle (gửi Tuấn kèm file zip)

- [ ] `env_versions.txt`, `git_commit.txt`, `checkpoint_sha256.txt`, `inference_params.json` đã có.
- [ ] Smoke test 20 ảnh: polygon 4 đỉnh, score ∈ [0,1], không out-of-bounds bất thường.
- [ ] `metrics.json`: P/R/Hmean ICDAR2015 test + latency + kiểm tra tất định.
- [ ] `visual_examples/`: 20 overlay đã xem bằng mắt (đặc biệt nhóm *worst*).
- [ ] `official_benchmark_availability.md`: đã điền phần "Đã kiểm tra" — 83,8 ghi **NOT VERIFIED**.
- [ ] `evaluation_type.txt` = `external` — không ghi là reproduction thành công.
- [ ] `dataset_manifest.tsv` sha256 đối chiếu với Tuấn (cùng bytes ICDAR2015 test).
- [ ] Nhóm đồng ý **bằng văn bản** rằng public-benchmark validation đủ cho phạm vi đồ án → mới sang Phase 2.